In [1]:
import os, json, textwrap, re, time
import requests

API_KEY  = os.getenv("OPENAI_API_KEY", "cse476")
API_BASE = os.getenv("API_BASE", "http://10.4.58.53:41701/v1")  
MODEL    = os.getenv("MODEL_NAME", "bens_model")              

def call_model_chat_completions(prompt: str,
                                system: str = "You are a helpful assistant. Reply with only the final answer—no explanation.",
                                model: str = MODEL,
                                temperature: float = 0.0,
                                timeout: int = 60) -> dict:
    """
    Calls an OpenAI-style /v1/chat/completions endpoint and returns:
    { 'ok': bool, 'text': str or None, 'raw': dict or None, 'status': int, 'error': str or None, 'headers': dict }
    """
    url = f"{API_BASE}/chat/completions"
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type":  "application/json",
    }
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user",   "content": prompt}
        ],
        "temperature": temperature,
        "max_tokens": 1024,
    }

    try:
        resp = requests.post(url, headers=headers, json=payload, timeout=timeout)
        status = resp.status_code
        hdrs   = dict(resp.headers)
        if status == 200:
            data = resp.json()
            text = data.get("choices", [{}])[0].get("message", {}).get("content", "")
            return {"ok": True, "text": text, "raw": data, "status": status, "error": None, "headers": hdrs}
        else:
            # try best-effort to surface error text
            err_text = None
            try:
                err_text = resp.json()
            except Exception:
                err_text = resp.text
            return {"ok": False, "text": None, "raw": None, "status": status, "error": str(err_text), "headers": hdrs}
    except requests.RequestException as e:
        return {"ok": False, "text": None, "raw": None, "status": -1, "error": str(e), "headers": {}}

In [5]:
import re
import time
import json
import concurrent.futures
from collections import Counter
from tqdm import tqdm  # pip install tqdm if needed

# =============================================================================
# SECTION 1: HELPERS & EXTRACTION
# =============================================================================

def extract_ans_xml(s):
    """
    Robustly extracts answer from <answer> tags, \boxed{}, or raw text.
    """
    if not s: return None
    s_clean = s.strip()

    # Priority 1: XML Tags <answer>...</answer>
    match = re.search(r"<answer>(.*?)</answer>", s, flags=re.DOTALL | re.IGNORECASE)
    if match: return match.group(1).strip()
    
    # Priority 2: LaTeX Boxed \boxed{...}
    match_box = re.search(r"\\boxed\{(.*?)\}", s)
    if match_box: return match_box.group(1).strip()

    # Priority 3: Fallback (if string is short/clean, it IS the answer)
    if len(s_clean) < 50 and "\n" not in s_clean: return s_clean
    return None

def vote_or_rescue(candidates, question, model):
    """
    Helper to perform voting or trigger a 'Rescue' call if no candidates exist.
    """
    valid_candidates = [c for c in candidates if len(c) < 100]
    
    if not valid_candidates:
        rescue_prompt = f"Question: {question}\nOutput ONLY the final answer value. No reasoning."
        rescue_r = call_model_chat_completions(
            rescue_prompt,
            system="You are an answer formatter. Output ONLY the answer.",
            model=model, temperature=0.0
        )
        raw_text = (rescue_r.get("text") or "").strip()
        return extract_ans_xml(raw_text) or raw_text

    most_common, count = Counter(valid_candidates).most_common(1)[0]
    return most_common

# =============================================================================
# SECTION 2: CORE REASONING TECHNIQUES
# =============================================================================

def solve_with_analogical(question, model):
    """
    Technique: Analogical Prompting.
    """
    user_prompt = question + "\n\nINSTRUCTION: Recall a similar problem pattern (e.g. Harmonic Mean, Apollonius), solve step-by-step, and output <answer>X</answer>."
    system_prompt = "You are a math tutor. Use analogical reasoning."
    r = call_model_chat_completions(user_prompt, system=system_prompt, model=model, temperature=0.0)
    return extract_ans_xml(r.get("text") or "")

def solve_with_cot(question, model, temperature=0.7):
    """
    Technique: Standard Chain of Thought.
    """
    r = call_model_chat_completions(
        question,
        system="You are a concise reasoning agent. Solve step-by-step. Wrap answer in <answer> tags.",
        model=model, temperature=temperature
    )
    return extract_ans_xml(r.get("text") or "")

def solve_with_cot_math_specialist(question, model, temperature=0.7):
    """
    Technique: Structured Math CoT (Algebra/Counting).
    """
    system_prompt = """You are a math expert.
Structure:
1. Analysis: Identify sub-field.
2. Calculation: Solve with clear algebra.
3. Verification: Check constraints (distinct digits, positive roots).
4. Final: <answer>VALUE</answer>.
"""
    r = call_model_chat_completions(question, system=system_prompt, model=model, temperature=temperature)
    return extract_ans_xml(r.get("text") or "")

def solve_with_geometry_specialist(question, model, temperature=0.7):
    """
    Technique: Geometry Specialist.
    Focuses on properties/theorems (Harmonic Mean, Bisected Diagonals).
    """
    system_prompt = """You are a Geometry Expert.
Guidelines:
1. List knowns.
2. CHECK PROPERTIES:
   - Triangles with parallel segments -> Harmonic Mean?
   - Quadrilaterals with area sums -> Bisected Diagonal?
   - Medians -> Apollonius?
3. Calculate.
4. Output <answer>VALUE</answer>.
"""
    r = call_model_chat_completions(question, system=system_prompt, model=model, temperature=temperature)
    return extract_ans_xml(r.get("text") or "")

# =============================================================================
# SECTION 3: DOMAIN-SPECIFIC SOLVERS
# =============================================================================

def solve_geometry(question, model):
    """
    GEOMETRY: Analogical + Geometry-Specific CoT
    """
    candidates = []
    ans = solve_with_analogical(question, model)
    if ans and ans != "FAIL" and len(ans) < 100: candidates.append(ans)

    for _ in range(5):
        ans = solve_with_geometry_specialist(question, model, temperature=0.7)
        if ans and ans != "FAIL" and len(ans) < 100: candidates.append(ans)
            
    return vote_or_rescue(candidates, question, model)

def solve_math(question, model):
    """
    GENERAL MATH: Analogical + Structured CoT
    """
    candidates = []
    ans = solve_with_analogical(question, model)
    if ans and ans != "FAIL" and len(ans) < 100: candidates.append(ans)

    for _ in range(5):
        ans = solve_with_cot_math_specialist(question, model, temperature=0.7)
        if ans and ans != "FAIL" and len(ans) < 100: candidates.append(ans)
            
    return vote_or_rescue(candidates, question, model)

def solve_coding(question, model):
    ans = solve_with_analogical(question, model)
    if ans and ans != "FAIL" and len(ans) < 800: return ans
    r = call_model_chat_completions(
        question,
        system="You are a python expert. Write clean code. Output result in <answer> tags.",
        model=model, temperature=0.2
    )
    return extract_ans_xml(r.get("text") or "") or "FAIL"

def solve_common_sense(question, model):
    candidates = []
    for _ in range(3):
        ans = solve_with_cot(question, model, temperature=0.7)
        if ans and ans != "FAIL": candidates.append(ans)
    return vote_or_rescue(candidates, question, model)

def solve_planning(question, model):
    user_prompt = question + "\n\nINSTRUCTION: List high-level steps first. Then execute. Output <answer>RESULT</answer>."
    r = call_model_chat_completions(user_prompt, system="You are a logistics planner.", model=model, temperature=0.0)
    return extract_ans_xml(r.get("text") or "") or "FAIL"

def solve_prediction(question, model):
    user_prompt = question + "\n\nINSTRUCTION: Make a specific prediction based on trends. Output <answer>PREDICTION</answer>."
    r = call_model_chat_completions(user_prompt, system="You are a forecaster.", model=model, temperature=0.7)
    return extract_ans_xml(r.get("text") or "") or "FAIL"

# =============================================================================
# SECTION 4: THE ROUTER AGENT (MAIN ENTRY POINT)
# =============================================================================

def run_agent(question, model=MODEL, domain=None):
    """
    Routes the question to the best solver based on domain.
    """
    # 1. MATH ROUTING (Geometry vs Algebra)
    if domain == 'math':
        q_lower = question.lower()
        # Keywords that signal the Geometry Specialist is needed
        if any(x in q_lower for x in ['triangle', 'quadrilateral', 'circle', 'polygon', 'geometry', 'parallel', 'diagonal']):
            return solve_geometry(question, model)
        return solve_math(question, model)
        
    # 2. OTHER DOMAINS
    elif domain == 'coding': return solve_coding(question, model)
    elif domain == 'planning': return solve_planning(question, model)
    elif domain == 'future_prediction': return solve_prediction(question, model)
    else: return solve_common_sense(question, model)

# =============================================================================
# SECTION 5: PARALLEL EVALUATION
# =============================================================================

def evaluate_single_test(t, model, grader_model):
    try:
        # 1. Extract Domain
        current_domain = t.get("id") 
        if current_domain not in ['math', 'coding', 'planning', 'future_prediction', 'common_sense']:
            current_domain = t.get("domain", "common_sense")

        # 2. Run Agent
        got = run_agent(t["prompt"], model=model, domain=current_domain)
        
        # 3. LLM Judge
        # We cast 'expected' and 'got' to strings to ensure the grader prompt doesn't crash
        grader_prompt = f"Question: {t['prompt']}\nCorrect: {str(t['expected'])}\nStudent: {str(got)}\nIs student correct? Reply YES or NO."
        
        r_grade = call_model_chat_completions(
            grader_prompt, system="Reply YES or NO.", model=grader_model, temperature=0.0
        )
        grade_text = (r_grade.get("text") or "").strip().upper()
        is_correct = "YES" in grade_text
        
        return {
            "id": current_domain,
            "expected": t["expected"],
            "got": got,
            "correct": is_correct,
            "error": None
        }
    except Exception as e:
        return {
            "id": t.get("id", "unknown"),
            "expected": t["expected"],
            "got": "ERROR",
            "correct": False,
            "error": str(e)
        }

def self_evaluate_tests_parallel(tests, verbose=True, model=MODEL, grader_model=MODEL, workers=10):
    print(f"Starting PARALLEL evaluation on {len(tests)} items with {workers} workers...")
    rows = []
    
    # Use ThreadPoolExecutor
    with concurrent.futures.ThreadPoolExecutor(max_workers=workers) as executor:
        future_to_test = {executor.submit(evaluate_single_test, t, model, grader_model): t for t in tests}
        
        try:
            for future in tqdm(concurrent.futures.as_completed(future_to_test), total=len(tests)):
                result = future.result()
                rows.append(result)
                
                if verbose and not result["correct"]:
                    # FIX: Cast to str() before slicing to avoid 'bool' or 'NoneType' errors
                    exp_str = str(result['expected'])
                    got_str = str(result['got'])
                    print(f"\n❌ [{result['id']}] Expected: {exp_str[:30]}... | Got: {got_str[:30]}...")
                    
        except KeyboardInterrupt:
            print("\nStopping early...")
            executor.shutdown(wait=False)

    correct_count = sum(1 for r in rows if r["correct"])
    print(f"\nFinal Score: {correct_count}/{len(rows)}")
    return rows

# =============================================================================
# SECTION 6: EXECUTION
# =============================================================================

with open('cse476_final_project_dev_data.json', 'r') as f:
    dev_data = json.load(f)

full_tests = []
for item in dev_data:
    full_tests.append({
        "id": item["domain"],
        "prompt": item["input"],
        "expected": item["output"]
    })

# Run with 20 workers (safe for text-only endpoints)
results = self_evaluate_tests_parallel(full_tests, workers=20)

Starting PARALLEL evaluation on 1000 items with 20 workers...


  0%|          | 1/1000 [00:51<14:15:19, 51.37s/it]


❌ [math] Expected: 306... | Got: 459...


  1%|          | 6/1000 [01:04<1:24:27,  5.10s/it] 


❌ [math] Expected: 72... | Got: 160...


...


❌ [common_sense] Expected: No, reindeer can't really fly... | Got: 3...

❌ [common_sense] Expected: Baby teeth placed under a pill... | Got: 3...


100%|█████████▉| 999/1000 [13:28<00:00,  2.37it/s]


❌ [common_sense] Expected: Ugly ducklings become ducks wh... | Got: 4...


100%|██████████| 1000/1000 [13:32<00:00,  1.23it/s]


Final Score: 400/1000


In [3]:
# agent_v2.py
# A stronger, budgeted agent with planner → sampler → verifier/refiner → vote.
# Assumes you already have:
#   - MODEL (string)
#   - call_model_chat_completions(prompt, system, model, temperature) -> {"text": "..."}
#
# Run: python agent_v2.py
# (expects cse476_final_project_dev_data.json in cwd for full eval; otherwise runs smoke tests)

import re, ast, operator, math, json, time, random
from fractions import Fraction
from collections import Counter, defaultdict
from typing import Optional, Union, Tuple, Dict, List
import concurrent.futures
from tqdm import tqdm

# =========================
# CONFIG
# =========================
class AgentConfig:
    def __init__(self):
        # Per-domain sampling settings
        self.temps_by_domain = {
            "math.geometry": [0.0, 0.2, 0.6, 0.9, 0.0],  # one extra deterministic pass
            "math.algebra":  [0.0, 0.2, 0.7, 0.9],
            "coding":        [0.0, 0.3],
            "planning":      [0.2, 0.6, 0.9],
            "commonsense":   [0.2, 0.6, 0.9],
            "other":         [0.2, 0.6],
        }
        # Max LLM calls per problem (hard cap)
        self.max_calls = 18
        # Early stop when we see this many identical normalized answers
        self.converge_k = 3
        # Verifier thresholds
        self.pass_score_tau = 0.75
        # Toggles for ablations
        self.use_router = True
        self.use_planner = True
        self.use_verifier = True
        self.use_refiner = True

CFG = AgentConfig()

# =========================
# UTIL: ANSWER EXTRACTION / NORMALIZATION
# =========================
ANSWER_TAG_RE = re.compile(r"<answer>(.*?)</answer>", flags=re.IGNORECASE | re.DOTALL)
BOXED_RE      = re.compile(r"\\boxed\{(.*?)\}")

def extract_answer(text: str) -> Optional[str]:
    if not text:
        return None
    m = ANSWER_TAG_RE.search(text)
    if m:
        return m.group(1).strip()
    m2 = BOXED_RE.search(text)
    if m2:
        return m2.group(1).strip()
    # fallback: try last non-empty line if it's short
    lines = [ln.strip() for ln in text.strip().splitlines() if ln.strip()]
    if lines:
        last = lines[-1]
        if len(last) < 80 and not last.lower().startswith(("analysis:", "solution:", "answer:")):
            return last.strip().strip(".")
    return None

# ---- SAFE ARITHMETIC EVAL (Python 3.8+ compatible) ----
_ALLOWED_BINOPS = {
    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
    ast.Div: operator.truediv, ast.FloorDiv: operator.floordiv,
    ast.Mod: operator.mod, ast.Pow: operator.pow
}
_ALLOWED_UNARYOPS = {ast.UAdd: operator.pos, ast.USub: operator.neg}

def _eval_ast(node):
    # numbers
    if isinstance(node, ast.Constant):
        if isinstance(node.value, (int, float)):
            return node.value
        raise ValueError("Only numeric constants allowed")
    if isinstance(node, ast.Num):  # for older Python ASTs
        return node.n

    # unary ops: +x, -x
    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_UNARYOPS:
        return _ALLOWED_UNARYOPS[type(node.op)](_eval_ast(node.operand))

    # binary ops: x+y, x-y, x*y, x/y, x//y, x%y, x**y
    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_BINOPS:
        left = _eval_ast(node.left)
        right = _eval_ast(node.right)
        return _ALLOWED_BINOPS[type(node.op)](left, right)

    # top-level expression node
    if isinstance(node, ast.Expression):
        return _eval_ast(node.body)

    # everything else is disallowed (calls, names, containers, etc.)
    raise ValueError("Disallowed expression")

def safe_eval_arith(expr: str) -> Union[float, int]:
    tree = ast.parse(expr, mode='eval')
    return _eval_ast(tree.body)

def normalize_candidate(ans_raw: str) -> Dict[str, Optional[Union[str, float, int, bool]]]:
    """
    Returns dict with:
      - 'raw', 'stripped', 'numeric_ok', 'value', 'units', 'is_integer', 'as_fraction'
    """
    out: Dict[str, Optional[Union[str, float, int, bool]]] = {
        "raw": ans_raw,
        "stripped": None,
        "numeric_ok": False,
        "value": None,
        "units": None,
        "is_integer": False,
        "as_fraction": None,
    }
    if ans_raw is None:
        return out
    s = ans_raw.strip()
    # simple units capture: trailing word chars or symbols
    m = re.match(r"^(.+?)\s*([a-zA-Z%$\u00A3\u20AC\u00A5]+)$", s)
    units = None
    if m:
        s_val, units = m.group(1).strip(), m.group(2)
    else:
        s_val = s

    # try fraction form like 'a/b'
    try:
        if re.fullmatch(r"[+-]?\d+\/[+-]?\d+", s_val):
            frac = Fraction(s_val)
            out["numeric_ok"] = True
            out["value"] = float(frac)
            out["as_fraction"] = str(frac)  # already lowest terms
            out["is_integer"] = frac.denominator == 1
            out["units"] = units
            out["stripped"] = s_val if units is None else f"{s_val} {units}"
            return out
    except Exception:
        pass

    # try plain int / float
    try:
        if re.fullmatch(r"[+-]?\d+(\.\d+)?", s_val):
            v = float(s_val)
            out["numeric_ok"] = True
            out["value"] = v
            out["is_integer"] = float(v).is_integer()
            out["as_fraction"] = str(Fraction(v).limit_denominator())
            out["units"] = units
            out["stripped"] = s_val if units is None else f"{s_val} {units}"
            return out
    except Exception:
        pass

    # try arithmetic expression
    try:
        v = safe_eval_arith(s_val)
        out["numeric_ok"] = True
        out["value"] = float(v)
        out["is_integer"] = float(v).is_integer()
        out["as_fraction"] = str(Fraction(v).limit_denominator())
        out["units"] = units
        out["stripped"] = (str(v) if units is None else f"{v} {units}")
        return out
    except Exception:
        pass

    # fallback: non-numeric answer string
    out["stripped"] = s
    out["units"] = units
    return out

def format_answer(value: str) -> str:
    return f"<answer>{value}</answer>"

# =========================
# PROMPTS
# =========================
ROUTER_SYS = "You are a domain router. Output ONLY one label from: {math.geometry, math.algebra, coding, planning, commonsense, other}."
PLANNER_SYS = "You create a short plan and constraints before solving."

PLANNER_USER_TMPL = """Problem:
{question}

Write:
1) Plan: ≤4 bullets.
2) Constraints/Units: key invariants, integer/positivity, unit expectations.
3) Key formulae (names only).
"""

SAMPLER_SYS_BASE = (
    "You are a careful solver. Use concise steps. Put the final numeric/string value in "
    "<answer>…</answer> at the end, and nowhere else. No other XML tags."
)

GEOM_SEED = "Consider: Similarity, Power of a Point, Pythagoras, Stewart/Apollonius, Angle Bisector, Harmonic division, area ratios with parallels."
ALG_SEED  = "Consider: factorization, parity/mod constraints, domain, boundary cases, extremum conditions."

VERIFIER_SYS = "You are a verifier. Output ONLY `OK` or `FIX:` followed by a minimal instruction."
VERIFIER_USER_TMPL = """Problem:
{question}
Spec:
{spec_short}
Candidate answer: {ans}

Check compliance (units, integer/positivity, domain). Return `OK` or `FIX: …`.
"""

REFINER_SYS = "You are a fixer. Repair only the issue. Keep it short. Output just one <answer>…</answer>."
RESCUE_SYS  = "You are an answer formatter. Output ONLY the final value in <answer>…</answer>."

# =========================
# ROUTER
# =========================
def route_domain(question: str) -> str:
    if not CFG.use_router:
        # heuristic fallback
        q = question.lower()
        if any(k in q for k in ["triangle", "circle", "polygon", "parallel", "diagonal", "chord", "tangent"]):
            return "math.geometry"
        if any(k in q for k in ["solve for", "equation", "integer", "remainder", "divides", "polynomial", "algebra"]):
            return "math.algebra"
        if any(k in q for k in ["code", "python", "algorithm", "runtime"]):
            return "coding"
        if any(k in q for k in ["plan", "schedule", "steps"]):
            return "planning"
        return "commonsense"
    # LLM router
    r = call_model_chat_completions(
        f"Problem:\n{question}\nLabel:",
        system=ROUTER_SYS, model=MODEL, temperature=0.0
    )
    label = (r.get("text") or "").strip().split()[0]
    return label if label in {"math.geometry","math.algebra","coding","planning","commonsense","other"} else "commonsense"

# =========================
# PLANNER (STEP-BACK)
# =========================
def build_spec_short(question: str) -> str:
    if not CFG.use_planner:
        return "Solve carefully; follow constraints and required units/format."
    r = call_model_chat_completions(
        PLANNER_USER_TMPL.format(question=question),
        system=PLANNER_SYS, model=MODEL, temperature=0.0
    )
    txt = (r.get("text") or "").strip()
    # Squash to ~200 chars to keep context light
    spec = re.sub(r"\s+", " ", txt)
    return (spec[:200] + "…") if len(spec) > 200 else spec

# =========================
# SAMPLER POOL
# =========================
def sampler_prompts(domain: str) -> List[str]:
    seeds = [SAMPLER_SYS_BASE]
    if domain == "math.geometry":
        seeds.append(SAMPLER_SYS_BASE + " " + GEOM_SEED)
    elif domain == "math.algebra":
        seeds.append(SAMPLER_SYS_BASE + " " + ALG_SEED)
    return seeds

def sample_candidates(question: str, spec_short: str, domain: str) -> List[str]:
    temps = CFG.temps_by_domain.get(domain, CFG.temps_by_domain["other"])
    seeds = sampler_prompts(domain)
    out: List[str] = []
    i = 0
    for t in temps:
        sys = seeds[min(i, len(seeds)-1)]
        r = call_model_chat_completions(
            f"Problem:\n{question}\n\nSpec (short): {spec_short}\nSolve. Put only the final value in <answer>…</answer>.",
            system=sys, model=MODEL, temperature=t
        )
        out.append(r.get("text") or "")
        i += 1
    return out

# =========================
# LOCAL CHECKS & SCORING
# =========================
def units_expected(spec_short: str) -> bool:
    return any(u in spec_short.lower() for u in ["unit", "meter", "cm", "km", "dollar", "$", "%", "degrees", "°"])

def requires_integer(spec_short: str) -> bool:
    s = spec_short.lower()
    return any(k in s for k in ["integer", "whole number", "count", "number of", "digits must be distinct"])

def local_score(norm: Dict[str, Optional[Union[str, float, int, bool]]], spec_short: str) -> float:
    score = 0.0
    # numeric validity
    if norm["numeric_ok"]:
        score += 0.35
    # integer requirement
    if requires_integer(spec_short):
        if norm["is_integer"]:
            score += 0.25
        else:
            score -= 0.15
    # units
    if units_expected(spec_short):
        if norm["units"] is not None or (norm["raw"] and re.search(r"(cm|m|km|%|°|degrees|\$|dollars)", str(norm["raw"]), re.I)):
            score += 0.2
        else:
            score -= 0.1
    # fraction in lowest terms is nice
    if norm["as_fraction"]:
        try:
            f = Fraction(str(norm["as_fraction"]))
            if Fraction(f.numerator, f.denominator) == f:  # lowest terms already
                score += 0.1
        except Exception:
            pass
    # bounded to [0,1]
    return max(0.0, min(1.0, score))

# =========================
# LLM VERIFIER & REFINER
# =========================
def verify_candidate(question: str, spec_short: str, ans_value: str) -> Tuple[bool, str]:
    if not CFG.use_verifier:
        return True, ""
    r = call_model_chat_completions(
        VERIFIER_USER_TMPL.format(question=question, spec_short=spec_short, ans=ans_value),
        system=VERIFIER_SYS, model=MODEL, temperature=0.0
    )
    txt = (r.get("text") or "").strip()
    if txt.startswith("OK"):
        return True, ""
    if txt.startswith("FIX:"):
        return False, txt
    # unknown -> treat as not ok with generic hint
    return False, "FIX: does not meet spec/format"

def refine_once(question: str, spec_short: str, current_ans: str, fix_msg: str) -> Optional[str]:
    if not CFG.use_refiner:
        return None
    r = call_model_chat_completions(
        f"Problem:\n{question}\nSpec:\n{spec_short}\nCurrent attempt: {current_ans}\nIssues:\n- {fix_msg}\nRepair.",
        system=REFINER_SYS, model=MODEL, temperature=0.0
    )
    return extract_answer(r.get("text") or "")

def rescue_answer(question: str) -> Optional[str]:
    r = call_model_chat_completions(
        f"Problem:\n{question}\nOutput ONLY the final value inside <answer>…</answer>—no steps.",
        system=RESCUE_SYS, model=MODEL, temperature=0.0
    )
    return extract_answer(r.get("text") or "")

# =========================
# CORE AGENT
# =========================
def solve_one(question: str, domain_hint: Optional[str] = None) -> str:
    calls_used = 0
    # 1) route
    domain = domain_hint or route_domain(question); calls_used += 1 if CFG.use_router and domain_hint is None else 0
    # 2) plan/spec
    spec_short = build_spec_short(question); calls_used += 1 if CFG.use_planner else 0
    # 3) sample
    raw_samples = sample_candidates(question, spec_short, domain)
    calls_used += len(raw_samples)

    # extract & normalize
    norms = []
    for txt in raw_samples:
        ans = extract_answer(txt)
        norms.append((ans, normalize_candidate(ans or "")))

    # early-stop convergence
    cnt = Counter([n[0] for n in norms if n[0]])
    for cand, c in cnt.items():
        if cand and c >= CFG.converge_k:
            return format_answer(cand)

    # 4) scoring + verifier
    scored = []
    for (ans, norm) in norms:
        if not ans:
            scored.append((ans, norm, 0.0, False, "no-answer"))
            continue
        local = local_score(norm, spec_short)
        ok, fix = verify_candidate(question, spec_short, ans)
        calls_used += 1 if CFG.use_verifier else 0
        # combine (simple)
        score = 0.6 * local + (0.4 if ok else 0.0)
        scored.append((ans, norm, score, ok, fix))

    # try single refiner on the top failing one (if budget allows)
    scored_sorted = sorted(scored, key=lambda x: x[2], reverse=True)
    if CFG.use_refiner and calls_used < CFG.max_calls and scored_sorted:
        best_ans, best_norm, best_score, best_ok, best_fix = scored_sorted[0]
        if not best_ok and best_fix and best_ans:
            repaired = refine_once(question, spec_short, best_ans, best_fix); calls_used += 1
            if repaired:
                repaired_norm = normalize_candidate(repaired)
                local = local_score(repaired_norm, spec_short)
                ok2, fix2 = verify_candidate(question, spec_short, repaired); calls_used += 1 if CFG.use_verifier else 0
                score2 = 0.6 * local + (0.4 if ok2 else 0.0)
                scored_sorted.append((repaired, repaired_norm, score2, ok2, fix2))
                cnt = Counter([s[0] for s in scored_sorted if s[0]])
                if cnt[repaired] >= CFG.converge_k:
                    return format_answer(repaired)

    # 5) rescue if nothing usable and we still have budget
    if calls_used < CFG.max_calls and all((s[0] or "") == "" for s in scored_sorted):
        resc = rescue_answer(question); calls_used += 1
        if resc:
            return format_answer(resc)

    # 6) calibrated vote / tie-break
    votes = Counter()
    info_by_ans: Dict[str, Tuple[float, bool]] = {}
    for ans, norm, sc, ok, fix in scored_sorted:
        if not ans:
            continue
        votes[ans] += 1
        info_by_ans[ans] = (sc, ok)
    if not votes:
        return format_answer("FAIL")

    modal, _ = votes.most_common(1)[0]
    tied = [a for a, c in votes.items() if c == votes[modal]]
    if len(tied) > 1:
        tied_sorted = sorted(tied, key=lambda a: (info_by_ans[a][0], info_by_ans[a][1]), reverse=True)
        winner = tied_sorted[0]
    else:
        winner = modal

    return format_answer(winner)

# =========================
# EVALUATION (parallel)
# =========================
def map_domain_hint(d: Optional[str]) -> Optional[str]:
    if not d: return None
    d = d.lower()
    if d == "math" or d == "algebra": return "math.algebra"
    if "geometry" in d: return "math.geometry"
    if "coding" in d: return "coding"
    if "planning" in d: return "planning"
    if "future_prediction" in d: return "commonsense"
    if "common_sense" in d: return "commonsense"
    return None

def evaluate_item(item: Dict[str, str]) -> Dict[str, Union[str, bool]]:
    q = item["input"]
    domain = item.get("domain") or item.get("id")
    try:
        ans = solve_one(q, domain_hint=map_domain_hint(domain))
        got = extract_answer(ans) or ""
        exp = item.get("output", "")
        correct = normalize_candidate(got)["stripped"] == normalize_candidate(exp)["stripped"]
        return {"id": str(domain), "expected": exp, "got": got, "correct": bool(correct)}
    except Exception as e:
        return {"id": "error", "expected": item.get("output",""), "got": str(e), "correct": False}

def run_eval(dev_path: str = "cse476_final_project_dev_data.json", workers: int = 10):
    with open(dev_path, "r") as f:
        data = json.load(f)
    tests = [{"id": x.get("domain","commonsense"), "input": x["input"], "output": x["output"]} for x in data]
    rows: List[Dict[str, Union[str, bool]]] = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=workers) as ex:
        futs = [ex.submit(evaluate_item, t) for t in tests]
        for fut in tqdm(concurrent.futures.as_completed(futs), total=len(futs)):
            r = fut.result()
            rows.append(r)
    correct = sum(1 for r in rows if r["correct"])
    print(f"\nFinal Score: {correct}/{len(rows)}")
    return rows

# =========================
# MAIN
# =========================
if __name__ == "__main__":
    # tiny smoke tests
    qs = [
        "If a triangle has sides 3,4,5, what is the area?",
        "Solve for integer x: 3x + 2 = 14",
        "You earn a 15% tip on $80. What is the total?"
    ]
    for q in qs:
        print(q)
        print(solve_one(q))
    # full eval (expects dev json)
    try:
        run_eval()
    except FileNotFoundError:
        print("Place cse476_final_project_dev_data.json in the working directory to run eval.")


If a triangle has sides 3,4,5, what is the area?
<answer>6</answer>
Solve for integer x: 3x + 2 = 14
<answer>4</answer>
You earn a 15% tip on $80. What is the total?
<answer>92</answer>


  2%|▏         | 20/1000 [01:33<34:48,  2.13s/it]  /var/folders/4c/9sgcv7bd2dl9ynrw5d3gd1mh0000gn/T/ipykernel_53317/2248949953.py:82: DeprecationWarning: ast.Num is deprecated and will be removed in Python 3.14; use ast.Constant instead
  if isinstance(node, ast.Num):  # for older Python ASTs
100%|██████████| 1000/1000 [20:33<00:00,  1.23s/it]


Final Score: 119/1000


In [1]:
import os, json, textwrap, re, time
import requests
from typing import List, Dict, Any, Optional
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
from tqdm import tqdm
import pandas as pd
from datetime import datetime

class ConcurrentReasoningAgent:
    def __init__(self, max_workers: int = 5):
        self.api_key = os.getenv("OPENAI_API_KEY", "cse476")
        self.api_base = os.getenv("API_BASE", "http://10.4.58.53:41701/v1")
        self.model = os.getenv("MODEL_NAME", "bens_model")
        self.max_calls_per_question = 20
        self.request_lock = threading.Lock()
        self.total_requests = 0
        self.max_workers = max_workers
        
    def call_model(self, prompt: str, system: str = None, temperature: float = 0.0) -> Dict:
        """Make API call with thread-safe rate limiting"""
        with self.request_lock:
            self.total_requests += 1
            
        if system is None:
            system = "You are a careful reasoning assistant. Think step by step and provide well-justified answers."
            
        url = f"{self.api_base}/chat/completions"
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json",
        }
        payload = {
            "model": self.model,
            "messages": [
                {"role": "system", "content": system},
                {"role": "user", "content": prompt}
            ],
            "temperature": temperature,
            "max_tokens": 512,
        }

        try:
            resp = requests.post(url, headers=headers, json=payload, timeout=60)
            
            if resp.status_code == 200:
                data = resp.json()
                text = data.get("choices", [{}])[0].get("message", {}).get("content", "")
                return {"ok": True, "text": text, "raw": data}
            else:
                return {"ok": False, "text": None, "error": f"HTTP {resp.status_code}"}
        except Exception as e:
            return {"ok": False, "text": None, "error": str(e)}
    
    # TECHNIQUE 1: Chain of Thought (CoT) Reasoning
    def chain_of_thought(self, problem: str) -> str:
        """Implement step-by-step reasoning"""
        cot_system = """You are a logical reasoning expert. Break down problems into clear steps.
        Think through each step carefully before concluding.
        Format your final answer as: <answer>final answer here</answer>"""
        
        prompt = f"""Please solve this problem step by step:

{problem}

Think through each logical step carefully. After your reasoning, provide your final answer wrapped in <answer> tags."""
        
        result = self.call_model(prompt, system=cot_system)
        return result["text"] if result["ok"] else None
    
    # TECHNIQUE 2: Self-Verification and Reflection
    def self_verification(self, problem: str, initial_answer: str) -> str:
        """Verify and refine the initial answer"""
        verification_system = """You are a careful verifier. Check if the given answer is correct and complete.
        Identify any potential errors or missing considerations."""
        
        prompt = f"""Problem: {problem}

Proposed Answer: {initial_answer}

Please verify this answer:
1. Check for logical errors
2. Identify any missing steps
3. Consider alternative perspectives
4. Provide a refined final answer if needed

After verification, provide your final answer wrapped in <answer> tags."""

        result = self.call_model(prompt, system=verification_system, temperature=0.1)
        return result["text"] if result["ok"] else initial_answer
    
    # TECHNIQUE 3: Multi-Perspective Reasoning
    def multi_perspective(self, problem: str, domain: str) -> str:
        """Approach problem from different perspectives based on domain"""
        perspectives = {
            "math": "mathematical precision and logical deduction",
            "common_sense": "practical knowledge and real-world understanding", 
            "planning": "systematic step-by-step action planning",
            "coding": "computational thinking and algorithmic approach",
            "future_prediction": "logical extrapolation and trend analysis"
        }
        
        perspective = perspectives.get(domain, "logical reasoning")
        
        system = f"""You are an expert in {perspective}. Use {perspective} to solve this problem accurately.
        Provide clear reasoning and wrap your final answer in <answer> tags."""
        
        result = self.call_model(problem, system=system, temperature=0.1)
        return result["text"] if result["ok"] else None
    
    # TECHNIQUE 4: Problem Decomposition
    def problem_decomposition(self, complex_problem: str) -> str:
        """Break down complex problems into simpler subproblems"""
        system = """You are an expert problem solver. Break complex problems into manageable parts.
        Solve each part systematically and combine the solutions."""
        
        prompt = f"""Please decompose and solve this complex problem:

{complex_problem}

Break it down into logical subproblems, solve each one, then combine the solutions.
Provide your final comprehensive answer wrapped in <answer> tags."""

        result = self.call_model(prompt, system=system)
        return result["text"] if result["ok"] else None
    
    # TECHNIQUE 5: Ensemble Reasoning (Multiple approaches)
    def ensemble_reasoning(self, problem: str, domain: str) -> str:
        """Use multiple techniques and choose the most consistent answer"""
        techniques = [
            lambda: self.chain_of_thought(problem),
            lambda: self.multi_perspective(problem, domain),
            lambda: self.problem_decomposition(problem)
        ]
        
        answers = []
        for technique in techniques[:2]:  # Use first 2 to save API calls
            try:
                result = technique()
                if result:
                    answer = self.extract_answer(result)
                    if answer and answer not in answers:
                        answers.append(answer)
            except Exception as e:
                continue
                
        # Return the most common answer, or first if no consensus
        if answers:
            return max(set(answers), key=answers.count)
        return "Unable to solve"
    
    def extract_answer(self, text: str) -> str:
        """Extract final answer from model response"""
        if not text:
            return ""
        
        # Try to extract from <answer> tags first
        match = re.search(r"<answer>(.*?)</answer>", text, re.DOTALL)
        if match:
            return match.group(1).strip()
        
        # Fallback: extract the last line or number
        lines = text.strip().split('\n')
        for line in reversed(lines):
            line = line.strip()
            if line and not line.startswith('Step') and not line.startswith('Reasoning'):
                # Look for numeric answers or final conclusions
                numbers = re.findall(r'-?\d+\.?\d*', line)
                if numbers:
                    return numbers[-1]
                if len(line) < 100:  # Reasonable length for final answer
                    return line
        
        return text.strip()
    
    def solve_problem(self, problem: str, domain: str = "unknown") -> str:
        """Main agent loop combining multiple techniques"""
        call_count_before = self.total_requests
        
        # Strategy selection based on problem characteristics and domain
        if domain == "planning" or any(word in problem.lower() for word in ['plan', 'steps', 'sequence', 'actions']):
            result = self.problem_decomposition(problem)
        elif domain == "math" or any(word in problem.lower() for word in ['calculate', 'solve for', 'equation']):
            result = self.chain_of_thought(problem)
        elif domain in ["common_sense", "future_prediction"]:
            result = self.multi_perspective(problem, domain)
        else:
            # Use ensemble for complex or unknown domains
            result = self.ensemble_reasoning(problem, domain)
            
        # Safety check: don't exceed max calls
        calls_used = self.total_requests - call_count_before
        if calls_used >= self.max_calls_per_question:
            print(f"Warning: Used {calls_used} API calls for one problem")
            
        return self.extract_answer(result) if result else "Unable to solve"

class ConcurrentEvaluator:
    def __init__(self, agent: ConcurrentReasoningAgent, max_workers: int = 5):
        self.agent = agent
        self.max_workers = max_workers
        
    def evaluate_single(self, test_case: Dict) -> Dict:
        """Evaluate agent on a single test case"""
        problem = test_case.get("input", "")
        expected = test_case.get("expected_output", "")
        domain = test_case.get("domain", "unknown")
        case_id = test_case.get("id", str(hash(problem)))
        
        start_time = time.time()
        requests_before = self.agent.total_requests
        
        try:
            answer = self.agent.solve_problem(problem, domain)
            api_calls_used = self.agent.total_requests - requests_before
        except Exception as e:
            answer = f"Error: {str(e)}"
            api_calls_used = 0
            
        end_time = time.time()
        
        is_correct = self.simple_match(expected, answer)
        
        return {
            "id": case_id,
            "domain": domain,
            "problem": problem[:200] + "..." if len(problem) > 200 else problem,
            "expected": expected,
            "answer": answer,
            "correct": is_correct,
            "time_taken": end_time - start_time,
            "api_calls": api_calls_used,
            "timestamp": datetime.now().isoformat()
        }
    
    def simple_match(self, expected: str, actual: str) -> bool:
        """Basic answer matching with normalization"""
        if not actual or "Unable to solve" in actual or "Error:" in actual:
            return False
            
        expected_norm = re.sub(r'[^\\w\\s]', ' ', expected.lower()).strip()
        actual_norm = re.sub(r'[^\\w\\s]', ' ', actual.lower()).strip()
        
        # For numeric answers, extract numbers
        expected_nums = re.findall(r'-?\d+\.?\d*', expected_norm)
        actual_nums = re.findall(r'-?\d+\.?\d*', actual_norm)
        
        if expected_nums and actual_nums:
            return expected_nums == actual_nums
            
        # For text answers, check containment or exact match
        return (expected_norm in actual_norm or 
                actual_norm in expected_norm or 
                expected_norm == actual_norm)
    
    def evaluate_concurrent(self, test_cases: List[Dict], progress_bar: bool = True) -> Dict:
        """Evaluate agent on multiple test cases concurrently"""
        results = []
        failed_cases = []
        
        print(f"Starting concurrent evaluation with {self.max_workers} workers...")
        print(f"Total test cases: {len(test_cases)}")
        
        with ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            # Submit all tasks
            future_to_case = {
                executor.submit(self.evaluate_single, case): case 
                for case in test_cases
            }
            
            # Process completed tasks with progress bar
            if progress_bar:
                futures = tqdm(as_completed(future_to_case), total=len(test_cases), desc="Evaluating")
            else:
                futures = as_completed(future_to_case)
                
            for future in futures:
                case = future_to_case[future]
                try:
                    result = future.result(timeout=300)  # 5 minute timeout per case
                    results.append(result)
                except Exception as e:
                    print(f"Failed to evaluate case: {str(e)}")
                    failed_cases.append({
                        "case": case,
                        "error": str(e)
                    })
                
                # Small delay to avoid overwhelming the API
                time.sleep(0.1)
        
        # Calculate comprehensive metrics
        summary = self._calculate_metrics(results)
        summary["failed_cases"] = len(failed_cases)
        
        return {
            "results": results,
            "failed_cases": failed_cases,
            "summary": summary
        }
    
    def _calculate_metrics(self, results: List[Dict]) -> Dict:
        """Calculate detailed evaluation metrics"""
        if not results:
            return {}
            
        total = len(results)
        correct = sum(1 for r in results if r["correct"])
        
        # Domain-wise accuracy
        domain_stats = {}
        for result in results:
            domain = result["domain"]
            if domain not in domain_stats:
                domain_stats[domain] = {"total": 0, "correct": 0}
            domain_stats[domain]["total"] += 1
            if result["correct"]:
                domain_stats[domain]["correct"] += 1
        
        domain_accuracy = {
            domain: {
                "accuracy": stats["correct"] / stats["total"],
                "total": stats["total"]
            }
            for domain, stats in domain_stats.items()
        }
        
        # Performance metrics
        avg_time = sum(r["time_taken"] for r in results) / total
        avg_api_calls = sum(r["api_calls"] for r in results) / total
        total_api_calls = sum(r["api_calls"] for r in results)
        
        return {
            "total_cases": total,
            "correct": correct,
            "accuracy": correct / total,
            "domain_accuracy": domain_accuracy,
            "avg_time_per_case": avg_time,
            "avg_api_calls_per_case": avg_api_calls,
            "total_api_calls": total_api_calls,
            "efficiency_score": correct / total_api_calls if total_api_calls > 0 else 0
        }

def main():
    # Initialize agent and evaluator
    max_workers = 50  # Conservative to avoid rate limiting
    agent = ConcurrentReasoningAgent(max_workers=max_workers)
    evaluator = ConcurrentEvaluator(agent, max_workers=max_workers)
    
    # Load full development data
    try:
        with open("cse476_final_project_dev_data.json", "r") as f:
            full_data = json.load(f)
        print(f"✅ Loaded {len(full_data)} development examples")
    except FileNotFoundError:
        print("❌ Development data file not found!")
        return
    except json.JSONDecodeError as e:
        print(f"❌ Error parsing JSON: {e}")
        return
    
    # Run evaluation on full dataset
    print("🚀 Starting full dataset evaluation...")
    start_time = time.time()
    
    evaluation_results = evaluator.evaluate_concurrent(full_data, progress_bar=True)
    
    end_time = time.time()
    total_duration = end_time - start_time
    
    # Print comprehensive results
    print(f"\n{'='*60}")
    print(f"📊 FULL DATASET EVALUATION RESULTS")
    print(f"{'='*60}")
    
    summary = evaluation_results["summary"]
    print(f"Total Cases Processed: {summary['total_cases']}")
    print(f"Correct Answers: {summary['correct']}")
    print(f"Overall Accuracy: {summary['accuracy']:.2%}")
    print(f"Failed Cases: {summary['failed_cases']}")
    print(f"Total Evaluation Time: {total_duration:.2f}s")
    print(f"Average Time per Case: {summary['avg_time_per_case']:.2f}s")
    print(f"Total API Calls: {summary['total_api_calls']}")
    print(f"Average API Calls per Case: {summary['avg_api_calls_per_case']:.1f}")
    print(f"Efficiency Score: {summary['efficiency_score']:.4f}")
    
    # Domain-wise breakdown
    print(f"\n📈 DOMAIN-WISE PERFORMANCE:")
    for domain, stats in summary['domain_accuracy'].items():
        print(f"  {domain:20} {stats['accuracy']:.2%} ({stats['total']} cases)")
    
    # Save detailed results
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    results_file = f"evaluation_results_{timestamp}.json"
    
    with open(results_file, "w") as f:
        json.dump(evaluation_results, f, indent=2)
    
    # Save CSV for easy analysis
    df = pd.DataFrame(evaluation_results["results"])
    csv_file = f"evaluation_results_{timestamp}.csv"
    df.to_csv(csv_file, index=False)
    
    print(f"\n💾 Results saved to:")
    print(f"  JSON: {results_file}")
    print(f"  CSV:  {csv_file}")
    
    # Show sample of incorrect answers for debugging
    incorrect = [r for r in evaluation_results["results"] if not r["correct"]]
    if incorrect:
        print(f"\n🔍 SAMPLE OF INCORRECT ANSWERS (showing first 3):")
        for i, result in enumerate(incorrect[:3]):
            print(f"\nExample {i+1} (Domain: {result['domain']}):")
            print(f"  Problem:  {result['problem'][:100]}...")
            print(f"  Expected: {result['expected']}")
            print(f"  Got:      {result['answer']}")
            print(f"  API Calls: {result['api_calls']}")

if __name__ == "__main__":
    main()

✅ Loaded 1000 development examples
🚀 Starting full dataset evaluation...
Starting concurrent evaluation with 50 workers...
Total test cases: 1000


Evaluating:   0%|          | 1/1000 [00:07<2:08:34,  7.72s/it]

Evaluating:   5%|▌         | 51/1000 [00:15<11:53,  1.33it/s] 

Evaluating:   5%|▌         | 53/1000 [00:15<06:39,  2.37it/s]

Evaluating:   6%|▌         | 56/1000 [00:15<03:22,  4.67it/s]

...

Evaluating:  90%|█████████ | 905/1000 [02:08<00:09,  9.54it/s]

Evaluating:  91%|█████████ | 907/1000 [02:08<00:09,  9.56it/s]

Evaluating:  91%|█████████ | 910/1000 [02:08<00:09,  9.56it/s]

Evaluating: 100%|██████████| 1000/1000 [02:18<00:00,  7.24it/s]


📊 FULL DATASET EVALUATION RESULTS
Total Cases Processed: 1000
Correct Answers: 955
Overall Accuracy: 95.50%
Failed Cases: 0
Total Evaluation Time: 138.23s
Average Time per Case: 6.33s
Total API Calls: 52925
Average API Calls per Case: 52.9
Efficiency Score: 0.0180

📈 DOMAIN-WISE PERFORMANCE:
  math                 100.00% (300 cases)
  coding               100.00% (100 cases)
  future_prediction    100.00% (100 cases)
  planning             100.00% (100 cases)
  common_sense         88.75% (400 cases)

💾 Results saved to:
  JSON: evaluation_results_20251123_013646.json
  CSV:  evaluation_results_20251123_013646.csv

🔍 SAMPLE OF INCORRECT ANSWERS (showing first 3):

Example 1 (Domain: common_sense):
  Problem:  Where in England was Dame Judi Dench born? Answer the question using the context.

 England is a cou...
  Expected: 
  Got:      Unable to solve
  API Calls: 2

Example 2 (Domain: common_sense):
  Problem:  From which country did Angola achieve independence in 1975? Answer the q